# 02 — Data Understanding and Cleaning

## Description

This notebook inspects and cleans the raw datasets.

The goal is to move from raw CSV files to clean, documented, analysis-ready datasets that can be used in the next notebooks for analytical modeling, feature engineering, data analysis and dashboard preparation.

Datasets used:

- `DataCoSupplyChainDataset.csv` — main order, customer, product, sales, logistics, and delivery dataset.
- `tokenized_access_logs.csv` — web access/clickstream dataset containing product/category page views.

Main tasks covered:

- load both raw CSV files
- inspect shape, schema, and column names
- standardize column names
- check missing values and duplicates
- parse date/time fields
- validate important business fields
- conduct cleaning and create quality-control features
- export cleaned datasets for downstream notebooks

## Table of Contents

## 1. Notebook Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

RAW_MAIN_PATH = RAW_DIR / 'DataCoSupplyChainDataset.csv'
RAW_LOGS_PATH = RAW_DIR / 'tokenized_access_logs.csv'

CLEAN_MAIN_PATH = PROCESSED_DIR / 'cleaned_supply_chain.csv'
CLEAN_LOGS_PATH = PROCESSED_DIR / 'cleaned_access_logs.csv'

PARQUET_MAIN_PATH = PROCESSED_DIR / 'cleaned_supply_chain.parquet'
PARQUET_LOGS_PATH = PROCESSED_DIR / 'cleaned_access_logs.parquet'

## 2. Load Raw Datasets

In [16]:
main_df = pd.read_csv(RAW_MAIN_PATH, encoding='latin1')
logs_df = pd.read_csv(RAW_LOGS_PATH)

## 3. Initial Dataset Exploration

### 3.1 Datasets Dimensions

In [18]:
dataset_shapes = pd.DataFrame({
    "dataset": ["transactions", "logs"],
    "rows": [main_df.shape[0], logs_df.shape[0]],
    "columns": [main_df.shape[1], logs_df.shape[1]]
})

dataset_shapes

,dataset,rows,columns
0,transactions,180519,53
1,logs,469977,8


### 3.2 Column-level schema summaries

In [19]:
def dataset_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
    'column_name': df.columns,
    'dtype': [df[col].dtype for col in df.columns],
    'non_null_count': [df[col].notna().sum() for col in df.columns],
    'null_count': [df[col].isna().sum() for col in df.columns],
    'unique_values': [df[col].nunique(dropna=True) for col in df.columns]
})

In [20]:
dataset_summary(main_df)

,column_name,dtype,non_null_count,null_count,unique_values
0,Type,str,180519,0,4
1,Days for shipping (real),int64,180519,0,7
2,Days for shipment (scheduled),int64,180519,0,4
3,Benefit per order,float64,180519,0,21998
4,Sales per customer,float64,180519,0,2927
5,Delivery Status,str,180519,0,4
6,Late_delivery_risk,int64,180519,0,2
7,Category Id,int64,180519,0,51
8,Category Name,str,180519,0,50
9,Customer City,str,180519,0,563


In [21]:
dataset_summary(logs_df)

,column_name,dtype,non_null_count,null_count,unique_values
0,Product,str,469977,0,76
1,Category,str,469977,0,33
2,Date,str,469977,0,160815
3,Month,str,469977,0,5
4,Hour,int64,469977,0,24
5,Department,str,469977,0,6
6,ip,str,469977,0,3340
7,url,str,469977,0,152


### 3.3 Preview

In [22]:
main_df.head(3)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [23]:
logs_df.head(3)

,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...


### 3.4 Initial Exploration Summary
- The main dataset contains 180,519 rows and 53 columns. The logs dataset contains 469977 rows and 8 columns.

- The main dataset combines customer, product, order, financial, and shipping attributes in a single denormalized table. Several columns appear to require further investigation, including fields with constant values, fully missing values, high missingness, and potentially redundant identifiers.